# Merge by ASIN

Outer-joins two Excel outputs on ASIN (e.g. the search-result dataset and the
product-detail dataset), normalises the key, reports duplicates, and labels each row
`BOTH`, `FILE1_ONLY` or `FILE2_ONLY` so gaps between the two sources are visible.

> See the [folder README](README.md) for the pipeline overview and the [disclaimer](../README.md#disclaimer).

In [ ]:
from datetime import datetime
import pandas as pd

try:
    from google.colab import files
    uploaded = files.upload()               # upload exactly two files
    FILE1, FILE2 = list(uploaded.keys())[:2]
except ImportError:
    FILE1, FILE2 = "../sample-data/amazon_products.csv", "../sample-data/amazon_details.csv"

read = lambda p: pd.read_csv(p) if p.endswith(".csv") else pd.read_excel(p)
df1, df2 = read(FILE1), read(FILE2)
print(f"{FILE1}: {len(df1):,} rows | {FILE2}: {len(df2):,} rows")


def asin_column(df, name):
    for col in df.columns:
        if str(col).strip().lower() == "asin":
            return col
    raise ValueError(f"No ASIN column in {name}: {list(df.columns)}")


def clean(df, name):
    col = asin_column(df, name)
    df = df.rename(columns={col: "ASIN"})
    df["ASIN"] = df["ASIN"].astype(str).str.strip().str.upper()
    df = df[df["ASIN"].notna() & ~df["ASIN"].isin(["", "NAN", "NONE"])]
    dups = df["ASIN"].duplicated(keep=False).sum()
    print(f"{name}: {'no duplicate ASINs' if dups == 0 else f'{dups} rows share an ASIN'}")
    return df


df1, df2 = clean(df1, FILE1), clean(df2, FILE2)

merged = pd.merge(df1, df2, on="ASIN", how="outer", suffixes=("_file1", "_file2"), indicator=True)
merged["merge_status"] = merged.pop("_merge").map(
    {"both": "BOTH", "left_only": "FILE1_ONLY", "right_only": "FILE2_ONLY"})
print(merged["merge_status"].value_counts().to_string())

merged = merged[["ASIN"] + [c for c in merged.columns if c != "ASIN"]]
out = f"asin_merged_{datetime.now():%y%m%d_%H%M%S}.xlsx"
merged.to_excel(out, index=False)
print("saved:", out)